# DS 208 · Week 9 — Working with data sources
## Files, SQL and APIs

Every code block from the lecture, in order, so you can run what you watched instead of
retyping it. The first slides use small tables typed into the cells; the **Case study**
cell loads the flood control data from the course API, so there is no file to download.

**How to use this.** Run the cells top to bottom the first time (Runtime → Run all). The
exercise cells list their tasks above the code: predict first, then run, then change one
thing and run again.

**Version differences.** The slides follow pandas 3. On pandas 2, text columns show as
`object` instead of `str`, and dates as `datetime64[ns]` instead of `datetime64[us]`.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

---
## The lecture, cell by cell

Headings carry the slide numbers shown in the deck's counter.


### Slide 7 — Files and SQL run on these slides


Expected output:

```
pandas version: 3.0.5
64-bit Python: True
city      str
pop     int64
dtype: object
```


In [ ]:
import io, sys

print("pandas version:", pd.__version__)
print("64-bit Python:", sys.maxsize > 2**32)

t = pd.read_csv(io.StringIO("city,pop\nCebu,964000"))
print(t.dtypes)

### Slide 10 — JSON is text resembling Python data


Expected output:

```
<class 'list'> <class 'dict'>
city      pop
0   Cebu   964000
1  Davao  1777000
```


In [ ]:
import json
import pandas as pd

txt = '[{"city": "Cebu", "pop": 964000}, {"city": "Davao", "pop": 1777000}]'
data = json.loads(txt)
print(type(data), type(data[0]))
print(pd.DataFrame(data))

### Slide 11 — Week 7’s eight cities, with their areas


Expected output:

```
(8, 3) (8, 2)
```


In [ ]:
import io

csv_text = """city,region,pop
Quezon City,Luzon,2960000
Manila,Luzon,1847000
Baguio,Luzon,366000
Cebu,Visayas,964000
Iloilo,Visayas,457000
Tacloban,Visayas,251000
Davao,Mindanao,1777000
Cagayan de Oro,Mindanao,728000"""
cities = pd.read_csv(io.StringIO(csv_text))
areas = pd.DataFrame({
    "city": ["Quezon City", "Manila", "Baguio", "Cebu",
             "Iloilo", "Tacloban", "Davao", "Cagayan de Oro"],
    "area_km2": [172, 42, 58, 315, 78, 202, 2444, 413]})

print(cities.shape, areas.shape)

### Slide 12 — pandas readers for each format


Expected output:

```
(8, 3)
(8, 3)
```


In [ ]:
# cities: the eight-city table from week 7
cities.to_csv("cities.csv", index=False)
cities.to_json("cities.json", orient="records")

print(pd.read_csv("cities.csv").shape)
print(pd.read_json("cities.json").shape)

### Slide 14 — The flood data, loaded from the course API


Expected output:

```
(34079, 15)
11.44 MB of CSV text
```


In [ ]:
import io, requests

url  = "https://portal.latarak.com/api/flood/export.csv"
text = requests.get(url, timeout=60).text
df   = pd.read_csv(io.StringIO(text))

print(df.shape)
print(round(len(text) / 1e6, 2), "MB of CSV text")

### Slide 16 — CSV files have no field for column types


Expected output:

```
float64
float64
```


In [ ]:
# CSV round-trip: types are guessed again on reading
df.to_csv("f.csv", index=False)
print(pd.read_csv("f.csv").budget.dtype)

# Parquet round-trip: types are stored in the file
df.to_parquet("f.parquet")
print(pd.read_parquet("f.parquet").budget.dtype)

### Slide 18 — pathlib for portable file paths


**Exercise.**

1. Predict what print(path) shows.
2. Run the cell.
3. Change "cities.csv" to "cities.json" and to_csv(path, index=False) to to_json(path, orient="records"). Is the file bigger or smaller?


In [ ]:
# cities: the eight-city table from week 7
from pathlib import Path

path = Path("data") / "cities.csv"
path.parent.mkdir(exist_ok=True)
cities.to_csv(path, index=False)

print(path, path.exists())
print(path.stat().st_size, "bytes")

### Slide 20 — Only one failure mode raises an error


**Exercise.**

1. Predict which of the two attempts raises an error.
2. Run the cell.
3. Replace "Cañete" with "Peñaranda" in both places. What changes?


In [ ]:
raw = "Cañete".encode("latin-1")      # bytes written by an old system
try:
    raw.decode("utf-8")                # read as utf-8 -> CRASH
except UnicodeDecodeError as e:
    print("UnicodeDecodeError:", e)

# utf-8 bytes read as latin-1 -> SILENT
print("Cañete".encode("utf-8").decode("latin-1"))

### Slide 21 — Specifying the encoding explicitly


Expected output:

```
b'\xef\xbb\xbft'
['Cañete', 'Peñaranda']
['CaÃ±ete', 'PeÃ±aranda']
```


In [ ]:
with open("towns.csv", "w", encoding="utf-8-sig") as f:   # as Excel saves it
    f.write("town\nCañete\nPeñaranda\n")

with open("towns.csv", "rb") as f:
    print(f.read(4))                   # the first bytes reveal the encoding

print(pd.read_csv("towns.csv", encoding="utf-8-sig").town.tolist())
print(pd.read_csv("towns.csv", encoding="latin-1").iloc[:, 0].tolist())

### Slide 22 — usecols: about 14× less memory


Timings measured on the lecture laptop; yours will differ, so compare the ratios:

```
59 ms, 13.6 MB, 15 columns
24 ms,  1.0 MB, 2 columns
```


In [ ]:
import time

def load(**kw):
    t = time.perf_counter()
    d = pd.read_csv(io.StringIO(text), **kw)
    ms = (time.perf_counter() - t) * 1000
    mb = d.memory_usage(deep=True).sum() / 1e6
    print(f"{ms:4.0f} ms, {mb:4.1f} MB, {d.shape[1]} columns")

load()
load(usecols=["region", "budget"])

### Slide 23 — chunksize returns an iterator


Expected output:

```
7 34079 1.6
2.0 MB in memory at most
```


In [ ]:
total, n, chunks, peak = 0, 0, 0, 0

for chunk in pd.read_csv(io.StringIO(text), chunksize=5000):
    total += chunk.budget.sum()
    n += len(chunk)
    chunks += 1
    peak = max(peak, chunk.memory_usage(deep=True).sum())

print(chunks, n, round(total / 1e12, 3))   # trillions of pesos
print(round(peak / 1e6, 1), "MB in memory at most")

### Slide 24 — json_normalize flattens nested records


**Exercise.**

1. Predict the column names of the result.
2. Run the cell.
3. Add a third record, {"id": 3, "loc": {"region": "NCR"}}, with no province. What does its row show?


In [ ]:
nested = [
  {"id": 1, "loc": {"region": "III", "prov": "Bulacan"}},
  {"id": 2, "loc": {"region": "V", "prov": "Albay"}},
]

print(pd.json_normalize(nested))

### Slide 28 — Query structure: SELECT, FROM, WHERE


**Exercise.**

1. Predict the rows the query returns, and their order.
2. Run the cell.
3. Change 'Visayas' to 'Luzon' and DESC to ASC. Which cities, in which order?


In [ ]:
# cities: the eight-city table from week 7
import sqlite3

con = sqlite3.connect(":memory:")       # a database in memory
cities.to_sql("cities", con, index=False)

query = """
SELECT city, pop
FROM cities
WHERE region = 'Visayas'
ORDER BY pop DESC;
"""
print(pd.read_sql(query, con))

### Slide 29 — read_sql returns a DataFrame


Expected output:

```
<class 'pandas.DataFrame'>
(8, 3)
```


In [ ]:
# con: the database from the previous slide
result = pd.read_sql("SELECT * FROM cities", con)

print(type(result))
print(result.shape)

### Slide 31 — A groupby written in SQL


Expected output:

```
region     n  total_b
0               Region III  5412    267.7
1  National Capital Region  3925    159.2
2                 Region V  2814    155.6
```


In [ ]:
import sqlite3
con = sqlite3.connect(":memory:")
df.to_sql("projects", con, index=False)

query = """
SELECT region, COUNT(*) AS n,
       ROUND(SUM(budget) / 1e9, 1) AS total_b
FROM projects
GROUP BY region
ORDER BY total_b DESC
LIMIT 3
"""
print(pd.read_sql(query, con))

### Slide 32 — The same groupby in pandas


Expected output:

```
n  total_b
region
Region III               5412    267.7
National Capital Region  3925    159.2
Region V                 2814    155.6
```


In [ ]:
top = (df.groupby("region")
         .agg(n=("contract_id", "count"),
              total=("budget", "sum"))
         .nlargest(3, "total"))
top["total_b"] = (top.total / 1e9).round(1)     # billions
print(top[["n", "total_b"]])

### Slide 33 — Parameterised queries with read_sql


Expected output:

```
5412
```


In [ ]:
# con: the database from the previous slide
r3 = pd.read_sql(
    "SELECT * FROM projects WHERE region = ?",
    con, params=["Region III"])
print(len(r3))

# NEVER: f"... WHERE region = '{user_input}'"
# that is an SQL injection, and a quote-mark bug

### Slide 34 — Saving results with to_sql


Expected output:

```
34079
n
0  34079
```


In [ ]:
written = df.to_sql("projects_copy", con, index=False,
                    if_exists="replace")
print(written)

print(pd.read_sql("SELECT COUNT(*) AS n FROM projects_copy", con))

### Slide 35 — JOIN: the SQL equivalent of merge


Expected output:

```
city      pop  area_km2
0  Quezon City  2960000       172
1       Manila  1847000        42
2       Baguio   366000        58
```


In [ ]:
# cities, areas: the eight cities and their land areas (week 7)
con = sqlite3.connect(":memory:")
cities.to_sql("cities", con, index=False)
areas.to_sql("areas", con, index=False)

query = """
SELECT c.city, c.pop, a.area_km2
FROM cities c
LEFT JOIN areas a ON c.city = a.city
"""
print(pd.read_sql(query, con).head(3))

### Slide 36 — The same join in pandas


Expected output:

```
city region      pop  area_km2
0  Quezon City  Luzon  2960000       172
1       Manila  Luzon  1847000        42
2       Baguio  Luzon   366000        58
```


In [ ]:
# cities, areas: the same two tables
both = cities.merge(areas, on="city", how="left")
print(both.head(3))

# INNER <-> how="inner"   LEFT <-> how="left"
# FULL  <-> how="outer"

### Slide 41 — requests.get and the status check


Expected output:

```
200
<class 'dict'>
34079 rows; 20/min anonymous, 200/min with the key
```


In [ ]:
import requests

r = requests.get("https://portal.latarak.com/api/flood", timeout=30)
print(r.status_code)                  # 200 = OK

info = r.json()                       # JSON text -> a Python dict
print(type(info))
print(info["rows"], "rows;", info["rate_limit"])

### Slide 43 — Converting a JSON response to a DataFrame


Expected output:

```
['next', 'page', 'pages', 'per_page', 'prev', 'results', 'total']
contract_id      region     status
0    22F00136    Region V  Completed
1    18BI0029   Region II  Completed
2    19CD0029  Region III  Completed
```


In [ ]:
import requests

url = "https://portal.latarak.com/api/flood/projects"
page = requests.get(url, params={"limit": 3}, timeout=30).json()

print(list(page))                     # the keys of the reply
first = pd.DataFrame(page["results"])
print(first[["contract_id", "region", "status"]])

### Slide 44 — The course API for practice


Expected output:

```
200
34079
amount_paid
```


In [ ]:
import requests

r = requests.get("https://portal.latarak.com/api/flood", timeout=30)
print(r.status_code)
print(r.json()["rows"])
print(r.json()["data_quality_warning"]["field"])

### Slide 45 — Large collections are returned one page at a time


Expected output:

```
300 341
```


In [ ]:
import requests

BASE = "https://portal.latarak.com"
HEAD = {"X-API-Key": "latarak-demo"}
url = BASE + "/api/flood/projects?limit=100"
rows, pages = [], 0

while url and pages < 3:               # all 34,079 rows = 341 pages
    r = requests.get(url, headers=HEAD, timeout=30).json()
    rows.extend(r["results"])
    url = r["next"]                    # None after the last page
    pages += 1

print(len(rows), r["pages"])

### Slide 46 — Rate limits, API keys and server load


Expected output:

```
200 18 regions
```


In [ ]:
import time, requests

BASE = "https://portal.latarak.com"
HEAD = {"X-API-Key": "latarak-demo"}      # the course's demo key

r = requests.get(BASE + "/api/flood/regions", headers=HEAD, timeout=30)
if r.status_code == 429:                  # too many requests
    time.sleep(int(r.headers["Retry-After"]))
    r = requests.get(BASE + "/api/flood/regions", headers=HEAD, timeout=30)

print(r.status_code, len(r.json()["results"]), "regions")

### Slide 47 — Handling network errors with try/except


Expected output:

```
status 404
```


In [ ]:
import requests

url = "https://portal.latarak.com/api/flood/projects/NO-SUCH-ID"
try:
    r = requests.get(url, timeout=10)
    r.raise_for_status()                # 4xx/5xx -> raise
    data = r.json()
except requests.Timeout:
    print("server too slow: using the cache")
except requests.HTTPError as e:
    print("status", e.response.status_code)
except ValueError:
    print("not JSON: got HTML?")

---

*Data: DPWH flood control projects 2016–2026, mirrored by BetterGov.ph from the DPWH
transparency portal, CC0-1.0 (public domain). Served by the course API at* `/api/flood`.
